# Sesión 6 · Gráficos que se entienden

Ya tenemos las tablas. Nadie lee tablas: la gente mira gráficos.

Usamos **Plotly Express**, que hace gráficos interactivos (se pasa el mouse y
aparece el dato) con una línea de código, y se exportan a HTML para pegarlos
en un correo o subirlos a una web.

In [1]:
import pandas as pd
import plotly.express as px

dengue = pd.read_csv("../03-pandas/dengue_limpio.csv", dtype={"ubigeo": str})
tabla = pd.read_csv("../03-pandas/tabla_distritos.csv", dtype={"ubigeo": str})
tasas = pd.read_csv("../04-scraping/tasas_departamento.csv")

dengue.shape, tabla.shape, tasas.shape

((172144, 9), (475, 8), (21, 5))

## 1. Barras: comparar categorías

La regla de Plotly Express: **un DataFrame, y le dices qué columna va en cada eje**.

In [2]:
top = tasas.sort_values("casos", ascending=False).head(10)

fig = px.bar(top, x="departamento", y="casos",
             title="Casos de dengue 2015-2021, por departamento")
fig.show()

### Barras horizontales: siempre mejor para nombres largos

Los nombres se leen sin girar la cabeza. `.sort_values` controla el orden.

In [3]:
fig = px.bar(top.sort_values("casos"), x="casos", y="departamento",
             orientation="h",
             title="Casos de dengue 2015-2021",
             labels={"casos": "Casos acumulados", "departamento": ""})
fig.show()

### El mismo gráfico, pero con la tasa

Compara este con el anterior: cambian los protagonistas. Ese contraste vale
más que cualquier explicación.

In [4]:
top_tasa = tasas.sort_values("tasa_100mil", ascending=False).head(10)

fig = px.bar(top_tasa.sort_values("tasa_100mil"), x="tasa_100mil", y="departamento",
             orientation="h",
             title="Dengue por cada 100 mil habitantes, 2015-2021",
             labels={"tasa_100mil": "Casos por 100 mil hab.", "departamento": ""},
             color="tasa_100mil", color_continuous_scale="Reds")
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 2. Líneas: cómo evoluciona algo en el tiempo

In [5]:
por_anio = dengue.groupby("anio")["casos"].sum().reset_index()

fig = px.line(por_anio, x="anio", y="casos", markers=True,
              title="Casos de dengue en el Perú, 2015-2021")
fig.show()

### Varias líneas a la vez: el parámetro `color`

`color` parte los datos en una línea por categoría. Es el parámetro más útil
de Plotly.

In [6]:
top5 = tasas.sort_values("casos", ascending=False).head(5)["departamento"].tolist()
evolucion = (
    dengue[dengue["departamento"].isin(top5)]
    .groupby(["anio", "departamento"])["casos"].sum().reset_index()
)

fig = px.line(evolucion, x="anio", y="casos", color="departamento", markers=True,
              title="Evolución del dengue en los 5 departamentos más afectados")
fig.show()

### Estacionalidad: por semana del año

Aquí se ve algo que la tabla escondía: el dengue tiene temporada.

In [7]:
por_semana = dengue.groupby(["anio", "semana"])["casos"].sum().reset_index()

fig = px.line(por_semana, x="semana", y="casos", color="anio",
              title="Estacionalidad del dengue por semana epidemiológica",
              labels={"semana": "Semana del año", "casos": "Casos"})
fig.show()

## 3. Dispersión: relacionar dos variables

¿Los distritos con más casos tienen más establecimientos de salud?

In [8]:
fig = px.scatter(tabla[tabla["casos_dengue"] > 0],
                 x="n_establecimientos", y="casos_dengue",
                 hover_name="distrito",
                 title="Casos de dengue vs. establecimientos de salud, por distrito")
fig.show()

### Con escala logarítmica y color

Cuando unos pocos puntos son gigantes y el resto está apelotonado, la escala
logarítmica deja ver el patrón.

In [9]:
fig = px.scatter(tabla[tabla["casos_dengue"] > 10],
                 x="n_establecimientos", y="casos_dengue",
                 color="departamento", size="casos_dengue",
                 hover_name="distrito", log_y=True,
                 title="Distritos: casos de dengue vs. oferta de salud",
                 labels={"n_establecimientos": "Establecimientos de salud",
                         "casos_dengue": "Casos (escala log)"})
fig.show()

## 4. Histograma: cómo se reparten los valores

In [10]:
fig = px.histogram(tabla[tabla["casos_dengue"] > 0], x="casos_dengue", nbins=50,
                   title="Distribución de casos por distrito")
fig.show()

La mayoría de distritos tiene pocos casos y unos pocos tienen muchísimos.
Eso explica por qué el promedio engaña y conviene mirar la mediana.

In [11]:
tabla["casos_dengue"].describe()

count      475.000000
mean       486.572632
std       1215.673913
min          0.000000
25%         11.000000
50%         70.000000
75%        407.000000
max      10989.000000
Name: casos_dengue, dtype: float64

## 5. Cajas: comparar distribuciones entre grupos

In [12]:
principales = tabla[tabla["departamento"].isin(top5)]

fig = px.box(principales, x="departamento", y="casos_dengue",
             title="Dispersión de casos entre distritos de un mismo departamento",
             log_y=True)
fig.show()

## 6. Facetas: un panel por categoría

`facet_col` repite el mismo gráfico para cada grupo. Sirve para comparar
formas, no niveles.

In [13]:
fig = px.line(evolucion, x="anio", y="casos", facet_col="departamento",
              facet_col_wrap=3, height=500,
              title="Cada departamento en su propio panel")
fig.update_yaxes(matches=None)     # cada panel con su propia escala
fig.show()

## 7. Que el gráfico se entienda solo

Un gráfico se manda por correo sin uno al lado explicándolo. Necesita título
que diga la conclusión, ejes con nombre y fuente.

In [14]:
fig = px.bar(
    top_tasa.sort_values("tasa_100mil"),
    x="tasa_100mil", y="departamento", orientation="h",
    color="tasa_100mil", color_continuous_scale="Reds",
    labels={"tasa_100mil": "Casos por 100 mil habitantes", "departamento": ""},
    title="Madre de Dios y Tumbes concentran la mayor incidencia de dengue",
)
fig.update_layout(
    coloraxis_showscale=False,
    template="plotly_white",
    title_font_size=18,
    margin=dict(l=10, r=10, t=60, b=60),
    annotations=[dict(
        text="Fuente: MINSA (casos 2015-2021) e INEI (Censo 2017). Elaboración propia.",
        showarrow=False, xref="paper", yref="paper",
        x=0, y=-0.18, font=dict(size=11, color="gray"),
    )],
)
fig.show()

## 8. Guardar el gráfico

`write_html` deja un archivo que se abre en cualquier navegador y **sigue
siendo interactivo**. No hace falta que el otro tenga Python.

In [15]:
fig.write_html("grafico_tasas.html")
print("Guardado. Ábrelo con doble clic.")

Guardado. Ábrelo con doble clic.


---
## Lo que hicimos

| Para | Código |
|---|---|
| Comparar categorías | `px.bar(df, x=, y=)` |
| Nombres largos | `orientation="h"` |
| Evolución temporal | `px.line(df, x=, y=, markers=True)` |
| Varias series | `color="columna"` |
| Relación entre variables | `px.scatter(df, x=, y=)` |
| Distribución | `px.histogram(df, x=)` |
| Comparar distribuciones | `px.box(df, x=, y=)` |
| Un panel por grupo | `facet_col=`, `facet_col_wrap=` |
| Aplanar valores extremos | `log_y=True` |
| Guardar | `fig.write_html("archivo.html")` |

### Tres reglas

1. **El título dice la conclusión**, no el contenido. "Madre de Dios lidera la
   incidencia" es mejor que "Casos por departamento".
2. **Ordena las barras.** Un gráfico de barras en orden alfabético desperdicia
   la mitad de su capacidad de comunicar.
3. **Pon la fuente.** Sin fuente, el gráfico no sirve para un informe.